In [1]:
import pandas as pd
from pathlib import Path
import numpy as np

VERBOSE = True
base_path = Path("data")
output_path = Path("my_data")

# 0. Load dataset


In [2]:
article_paths = sorted(base_path.glob("articles_file_[0-9][0-9].tsv"))
comment_paths = sorted(base_path.glob("comments_file_*_[0-9][0-9].tsv"))

if not article_paths:
    raise FileNotFoundError(f"No article files found in {base_path}")
if not comment_paths:
    raise FileNotFoundError(f"No comment files found in {base_path}")

# Load articles
articles_df = pd.concat((
        pd.read_csv(
            article_path,
            sep="\t",
            dtype={"Article_Id": "string", "Topic_1": "string", "Topic_2": "string", "Topic_3": "string"},
            keep_default_na=False,
            usecols=["Article_Id", "Topic_1", "Topic_2", "Topic_3"],
        )for article_path in article_paths
    ),ignore_index=True,
)

# Load comments
comments_df = pd.concat((
        pd.read_csv(
            comment_path,
            sep="\t",
            dtype={"Comment_Id": "string", "Article_Id": "string", "Author_Id": "string"},
            keep_default_na=False,
            usecols=["Comment_Id", "Article_Id", "Author_Id"],
        )for comment_path in comment_paths
    ),ignore_index=True,
)

if VERBOSE:
    # Article counts
    print(f"Number of articles: {len(articles_df):,}")
    print(f"Memory usage of articles: {articles_df.memory_usage(deep=True).sum() / 1024 ** 2:,.2f} MB")
    display(articles_df.head())

    # Comment counts
    print(f"Number of comments: {len(comments_df):,}")
    print(f"Number of unique authors: {comments_df['Author_Id'].nunique():,}")
    print(f"Memory usage of comments: {comments_df.memory_usage(deep=True).sum() / 1024 ** 2:,.2f} MB")
    display(comments_df.head())


Number of articles: 586,942
Memory usage of articles: 129.44 MB


,Article_Id,Topic_1,Topic_2,Topic_3
0,1358304331575,kultur,buehne,theater
1,1385172280895,sport,fussball,premierleague
2,1369362748065,wirtschaft,unternehmen,
3,1381372201348,international,europa,serbien
4,1381371265239,sport,fussball,europaleague


Number of comments: 75,644,850
Number of unique authors: 211,109
Memory usage of comments: 13,772.57 MB


,Comment_Id,Author_Id,Article_Id
0,459b1fc3fdb3b71c,bc7b5ef481527026,1356426507120
1,e729c1fa811753d8,c77eeefcc349e3d9,1356426507120
2,0769323cfc28e5bb,7d230b8728d8107a,1356426507120
3,e9af67b3a58039aa,fb90da25add7eb66,1356426507120
4,fdb4c0f7ca17d502,0c26f7be05e52b11,1356426507120


# 1. Create topic dataframe

In [3]:
# One row per unique topic path at each hierarchy level; parent counts include descendants
topic_columns = ["Topic_1", "Topic_2", "Topic_3"]
article_topics = articles_df[["Article_Id", *topic_columns]].copy()
article_topics[topic_columns] = article_topics[topic_columns].fillna("").apply(
    lambda column: column.str.strip()
)
article_topics = article_topics.drop_duplicates()
if article_topics["Article_Id"].duplicated().any():
    raise ValueError("An Article_Id has conflicting topic paths; resolve these before counting users.")

# Keep IDs as strings as loaded above. Blank authors do not represent users.¡
authors = comments_df["Author_Id"].where(comments_df["Author_Id"].notna() & comments_df["Author_Id"].str.strip().ne(""))
unmatched_comments = (~comments_df["Article_Id"].isin(article_topics["Article_Id"])).sum()
if VERBOSE:
    print(f"Comments with unmatched Article_Id (excluded): {unmatched_comments:,}\n")

level_frames = []
for level in (1, 2, 3):
    prefix = topic_columns[:level]
    eligible = article_topics.loc[article_topics[prefix].ne("").all(axis=1)].copy()
    # Integer codes avoid joining topic strings onto the full comments table.
    eligible["_topic_id"] = eligible.groupby(prefix, sort=False).ngroup()
    article_to_topic = eligible.set_index("Article_Id")["_topic_id"]
    comment_topics = comments_df["Article_Id"].map(article_to_topic)
    counts = authors.groupby(comment_topics, sort=False).nunique()

    paths = eligible[[*prefix, "_topic_id"]].drop_duplicates().copy()
    paths["Unique_users"] = paths["_topic_id"].map(counts).fillna(0).astype("int64")
    for column in topic_columns[level:]:
        paths[column] = ""
    paths["Topic_level"] = level
    level_frames.append(paths.drop(columns="_topic_id"))

# Include paths with no comments, with Unique_users = 0.
topics_df = pd.concat(level_frames, ignore_index=True)
parents_with_topic3 = pd.MultiIndex.from_frame(
    article_topics.loc[article_topics[topic_columns].ne("").all(axis=1), topic_columns[:2]]
    .drop_duplicates()
)
has_topic3_children = pd.MultiIndex.from_frame(topics_df[topic_columns[:2]]).isin(
    parents_with_topic3
)
# Topic_1 is always a node; Topic_2 is a leaf only when it has no Topic_3 children.
topics_df["Is_Leaf"] = False
topics_df.loc[
    topics_df["Topic_level"].eq(3)
    | (topics_df["Topic_level"].eq(2) & ~has_topic3_children),
    "Is_Leaf",
] = True
topics_df = (
    topics_df[[*topic_columns, "Topic_level", "Is_Leaf", "Unique_users"]]
    .sort_values(topic_columns)
    .reset_index(drop=True)
)

# Reuse a stable integer topic ID for article/comment selection below.
topics_df.index.name = "Topic_id"
article_topic_ids = pd.MultiIndex.from_frame(article_topics[topic_columns]).map(
    topics_df.reset_index().set_index(topic_columns)["Topic_id"]
)
article_to_topic_id = pd.Series(article_topic_ids, index=article_topics["Article_Id"])
article_topic_ids = articles_df["Article_Id"].map(article_to_topic_id)
comment_topic_ids = comments_df["Article_Id"].map(article_to_topic_id)

if VERBOSE:
    # Topic counts
    topic_level_counts = topics_df["Topic_level"].value_counts()
    print(f"Unique Topic_1 tags: {topic_level_counts.get(1, 0):,}")
    print(f"Unique Topic_2 paths (Topic_1, Topic_2): {topic_level_counts.get(2, 0):,}")
    print(f"\tUnique Topic_2 leaves: {len(topics_df.loc[topics_df['Topic_level'].eq(2) & topics_df['Is_Leaf'], 'Topic_2']):,}")
    print(f"Unique Topic_3 paths (Topic_1, Topic_2, Topic_3): {topic_level_counts.get(3, 0):,}")
    print(f"Total unique topic paths: {len(topics_df):,}")
    display(topics_df.head())


Comments with unmatched Article_Id (excluded): 382,167

Unique Topic_1 tags: 25
Unique Topic_2 paths (Topic_1, Topic_2): 307
	Unique Topic_2 leaves: 204
Unique Topic_3 paths (Topic_1, Topic_2, Topic_3): 1,775
Total unique topic paths: 2,107


,Topic_1,Topic_2,Topic_3,Topic_level,Is_Leaf,Unique_users
Topic_id,,,,,,
0,datenjournalismus,,,1,False,3909
1,diestandard,,,1,False,36590
2,diestandard,alltag,,2,False,16575
3,diestandard,alltag,dieessstoerungen,3,True,114
4,diestandard,alltag,diekultur,3,True,5150


# 2. Keep only Topic_3 and fallback Topic_2


In [4]:
topics_leaf_df = topics_df.loc[topics_df["Is_Leaf"]]
articles_leaf_df = articles_df.loc[article_topic_ids.isin(topics_leaf_df.index)].copy()
comments_leaf_df = comments_df.loc[comment_topic_ids.isin(topics_leaf_df.index)].copy()

if VERBOSE:
    print(f"Articles retained: {len(articles_leaf_df):,}")
    print(f"Comments for retained articles: {len(comments_leaf_df):,}")
    print(f"Unique users retained: {comments_leaf_df['Author_Id'].nunique():,}")
    print(f"Unique Topic_2 paths retained (Topic_1, Topic_2): {topics_leaf_df[['Topic_1', 'Topic_2']].drop_duplicates().shape[0]:,}")
    print(f"Unique Topic_3 paths retained (Topic_1, Topic_2, Topic_3): {topics_leaf_df['Topic_level'].eq(3).sum():,}")

Articles retained: 354,902
Comments for retained articles: 49,844,542
Unique users retained: 179,438
Unique Topic_2 paths retained (Topic_1, Topic_2): 307
Unique Topic_3 paths retained (Topic_1, Topic_2, Topic_3): 1,775


Alternative filter: only topic 3 articles

In [5]:
# topic3_df = topics_df.loc[topics_df["Topic_level"].eq(3)]
# articles_topic3 = articles_df.loc[article_topic_ids.isin(topic3_df.index)].copy()
# comments_topic3 = comments_df.loc[comment_topic_ids.isin(topic3_df.index)].copy()

# print(f"Topic_3 articles: {len(articles_topic3):,}")
# print(f"Comments for Topic_3 articles: {len(comments_topic3):,}")
# print(f"Unique users in Topic_3 comments: {comments_topic3['Author_Id'].nunique():,}")
# print(f"Unique Topic_3 paths (Topic_1, Topic_2, Topic_3): {len(topic3_df):,}")


Alternative fitler: only topic 2 leaves

In [6]:
# fallback_topics_df = topics_df.loc[topics_df["Topic_level"].eq(2) & topics_df["Is_Leaf"]]
# articles_fallback_topic2 = articles_df.loc[article_topic_ids.isin(fallback_topics_df.index)].copy()
# comments_fallback_topic2 = comments_df.loc[comment_topic_ids.isin(fallback_topics_df.index)].copy()

# print(f"Fallback Topic_2 articles: {len(articles_fallback_topic2):,}")
# print(f"Comments for fallback Topic_2 articles: {len(comments_fallback_topic2):,}")
# print(f"Unique users in fallback Topic_2 comments: {comments_fallback_topic2['Author_Id'].nunique():,}")
# print(f"Topic_2 paths without Topic_3 descendants: {len(fallback_topics_df):,}")


# 3. Keep top 1,000 topics by unique user count
Rank leaf topics using the distinct-user counts already computed in `topics_df`.


In [8]:
ranked_topics_df = topics_leaf_df.loc[topics_leaf_df["Unique_users"].gt(0)].sort_values(
    ["Unique_users", "Topic_1", "Topic_2", "Topic_3"],
    ascending=[False, True, True, True]
)

topics_top_1000 = ranked_topics_df.head(1_000)
articles_top_1000 = articles_df.loc[article_topic_ids.isin(topics_top_1000.index)].copy()
comments_top_1000 = comments_df.loc[comment_topic_ids.isin(topics_top_1000.index)].copy()

# Add topic IDs to topics df and articles df
topics_top_1000.insert(0, "Topic_Id", np.arange(len(topics_top_1000)))
articles_top_1000 = (articles_top_1000.merge(
        topics_top_1000[["Topic_Id", "Topic_1", "Topic_2", "Topic_3"]],
        on=["Topic_1", "Topic_2", "Topic_3"], how="left", validate="many_to_one")
)
articles_top_1000["Topic_Id"] = articles_top_1000["Topic_Id"].astype("int64")

if VERBOSE:
    print(f"Topic paths retained: {len(topics_top_1000):,}")
    print(f"Unique users retained: {comments_top_1000["Author_Id"].nunique():,}")
    print(f"Articles retained: {len(articles_top_1000):,}")
    print(f"Comments retained: {len(comments_top_1000):,}")
    print(f"Level 2 topic paths: {topics_top_1000['Topic_level'].eq(2).sum():,}")
    print(f"Level 3 topic paths: {topics_top_1000['Topic_level'].eq(3).sum():,}")


Topic paths retained: 1,000
Unique users retained: 176,892
Articles retained: 321,342
Comments retained: 48,871,807
Level 2 topic paths: 87
Level 3 topic paths: 913


In [9]:
display(topics_top_1000.head())
display(articles_top_1000.head())
display(comments_top_1000.head())

ranked_topics_df.to_csv(Path(output_path) / "ranked_topics.csv", index=False)
topics_top_1000.to_csv(Path(output_path) / "topics_top_1000.csv", index=False)
articles_top_1000.to_csv(Path(output_path) / "articles_top_1000.csv", index=False)
comments_top_1000.to_csv(Path(output_path) / "comments_top_1000.csv", index=False)

,Topic_Id,Topic_1,Topic_2,Topic_3,Topic_level,Is_Leaf,Unique_users
Topic_id,,,,,,,
112,0,diskurs,debatten,kommentare-der-anderen,3,True,46856
124,1,diskurs,kolumnen,rauscher,3,True,40494
860,2,international,amerika,usa,3,True,39512
926,3,international,europa,deutschland,3,True,34147
758,4,inland,parteien,fpoe,3,True,34041


,Article_Id,Topic_1,Topic_2,Topic_3,Topic_Id
0,1358304331575,kultur,buehne,theater,351
1,1385172280895,sport,fussball,premierleague,175
2,1381372201348,international,europa,serbien,320
3,1381371265239,sport,fussball,europaleague,155
4,1360681680482,diskurs,kommentare,frey,20


,Comment_Id,Author_Id,Article_Id
0,459b1fc3fdb3b71c,bc7b5ef481527026,1356426507120
1,e729c1fa811753d8,c77eeefcc349e3d9,1356426507120
2,0769323cfc28e5bb,7d230b8728d8107a,1356426507120
3,e9af67b3a58039aa,fb90da25add7eb66,1356426507120
4,fdb4c0f7ca17d502,0c26f7be05e52b11,1356426507120
